# ML in Cybersecurity: Suspicious Login Detector

**Workshop:** Can AI Detect a Suspicious Login?  
**Event date:** 4 November 2026  
**Time:** 2:00 PM–4:00 PM  
**Location:** Room C221

You will train a simple Machine Learning model that reviews **fictional login attempts** and predicts whether each login appears **normal** or **suspicious**.

> **Key message:** A suspicious prediction is a risk signal, not proof that a person is an attacker.

This is a classroom prototype. Do not connect it to a real authentication system.

## Workshop roadmap

1. Generate synthetic login data.
2. Explore features and labels.
3. Train a Decision Tree.
4. Evaluate accuracy and mistakes.
5. Visualise the model.
6. Test fictional login scenarios.
7. Try to confuse the model.

## 1. Import the libraries

`pandas` handles tables, `NumPy` generates synthetic values, `Matplotlib` creates charts, and `scikit-learn` provides the ML tools.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

print("Setup complete!")

## 2. Generate synthetic login data

Every row is one fictional login attempt. No real passwords, accounts, IP addresses or locations are used.

- `login_hour`: 0–23
- `failed_attempts`: recent incorrect password attempts
- `new_device`: 1 for new, 0 for recognised
- `distance_from_home_km`: fictional distance from the usual location
- `vpn_used`: 1 for yes, 0 for no
- `attempts_last_10_minutes`: recent login frequency

In [ ]:
np.random.seed(42)
number_of_logins = 1000

data = pd.DataFrame({
    "login_hour": np.random.randint(0, 24, number_of_logins),
    "failed_attempts": np.random.randint(0, 10, number_of_logins),
    "new_device": np.random.randint(0, 2, number_of_logins),
    "distance_from_home_km": np.random.randint(0, 1000, number_of_logins),
    "vpn_used": np.random.randint(0, 2, number_of_logins),
    "attempts_last_10_minutes": np.random.randint(1, 15, number_of_logins)
})

data.head()

## 3. Create workshop labels

- `0` = normal
- `1` = suspicious

For this workshop, points are assigned to combinations of unusual behaviour. Real labels would need verified incident outcomes.

**Important:** VPN use, travel, a new device or late-night access does not prove malicious behaviour.

In [ ]:
risk_points = (
    (data["failed_attempts"] >= 5).astype(int) * 3
    + (data["new_device"] == 1).astype(int) * 1
    + (data["distance_from_home_km"] >= 300).astype(int) * 2
    + (data["vpn_used"] == 1).astype(int) * 1
    + (data["attempts_last_10_minutes"] >= 8).astype(int) * 3
    + data["login_hour"].isin([0, 1, 2, 3, 4]).astype(int) * 1
)

data["suspicious"] = (risk_points >= 6).astype(int)
data.head(10)

## 4. Inspect the dataset

Look for differences, then discuss whether genuine travellers or VPN users could be incorrectly flagged.

In [ ]:
print("Number of login attempts:", len(data))
print()
print("Class counts:")
print(data["suspicious"].value_counts().rename(index={0: "Normal", 1: "Suspicious"}))
print()
print("Class percentages:")
print((data["suspicious"].value_counts(normalize=True) * 100).round(1).rename(index={0: "Normal", 1: "Suspicious"}))

In [ ]:
comparison = data.groupby("suspicious").mean(numeric_only=True).round(2)
comparison.index = ["Normal", "Suspicious"]
comparison

### Discuss

- Which signals differ most?
- Could a genuine traveller be flagged?
- Is a VPN automatically dangerous?
- What useful context is missing?

## 5. Separate features and labels

`X` contains the evidence. `y` contains the expected answer.

In [ ]:
feature_names = [
    "login_hour", "failed_attempts", "new_device",
    "distance_from_home_km", "vpn_used", "attempts_last_10_minutes"
]
X = data[feature_names]
y = data["suspicious"]

print("Feature table shape:", X.shape)
print("Label shape:", y.shape)
display(X.head())

## 6. Split the data

The model learns from 80% and is checked on 20%. Training data is the revision material; test data is the exam.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training examples:", len(X_train))
print("Testing examples:", len(X_test))

## 7. Train the Decision Tree

The model learns a sequence of yes/no questions. Limiting the tree helps reduce overfitting and makes it easier to explain.

In [ ]:
model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=42)
model.fit(X_train, y_train)
print("Model trained successfully!")

## 8. Evaluate the model

Accuracy is useful, but it does not reveal what kinds of mistakes were made.

In [ ]:
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print("Accuracy:", round(accuracy * 100, 2), "%")

In [ ]:
print(classification_report(
    y_test, predictions,
    target_names=["Normal", "Suspicious"],
    zero_division=0
))

- **Precision:** When the model flags a login, how often is it correct?
- **Recall:** How many labelled suspicious logins did it detect?
- **False positive:** A normal login is incorrectly flagged.
- **False negative:** A suspicious login is incorrectly allowed.

In [ ]:
cm = confusion_matrix(y_test, predictions)
ConfusionMatrixDisplay(cm, display_labels=["Normal", "Suspicious"]).plot(cmap="Blues", values_format="d")
plt.title("Suspicious Login Confusion Matrix")
plt.show()

tn, fp, fn, tp = cm.ravel()
print("True negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)

### Security meaning

False positives create inconvenience and alert fatigue. False negatives may allow suspicious access. Both matter.

## 9. Visualise the Decision Tree

Follow one route through the tree rather than reading every branch.

In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(model, feature_names=feature_names, class_names=["Normal", "Suspicious"],
          filled=True, rounded=True, fontsize=9)
plt.title("How the Suspicious Login Detector Makes Decisions")
plt.show()

## 10. Inspect feature importance

Importance describes this fitted model and dataset. It does not prove causation.

In [ ]:
importance = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

display(importance)
plt.figure(figsize=(9, 5))
plt.barh(importance["feature"], importance["importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Which Features Matter Most to This Model?")
plt.gca().invert_yaxis()
plt.show()

## 11. Build the login checker

The model score is not proof or a calibrated probability that a person is an attacker.

In [ ]:
def check_login(login_hour, failed_attempts, new_device,
                distance_from_home_km, vpn_used, attempts_last_10_minutes):
    login = pd.DataFrame([{
        "login_hour": login_hour,
        "failed_attempts": failed_attempts,
        "new_device": new_device,
        "distance_from_home_km": distance_from_home_km,
        "vpn_used": vpn_used,
        "attempts_last_10_minutes": attempts_last_10_minutes
    }])

    prediction = model.predict(login)[0]
    suspicious_score = model.predict_proba(login)[0][1] * 100
    result = "SUSPICIOUS" if prediction == 1 else "NORMAL"

    if suspicious_score < 30:
        response = "Allow login and continue monitoring"
    elif suspicious_score < 70:
        response = "Request additional verification"
    else:
        response = "Request MFA and notify the user"

    print("-" * 42)
    print("LOGIN SECURITY ANALYSIS")
    print("-" * 42)
    print("Prediction:", result)
    print("Suspicious score:", round(suspicious_score, 2), "%")
    print("Illustrative response:", response)

    return {"prediction": result,
            "suspicious_score": round(suspicious_score, 2),
            "illustrative_response": response}

### Scenario A: Clearly normal

In [ ]:
normal_result = check_login(
    login_hour=14, failed_attempts=0, new_device=0,
    distance_from_home_km=4, vpn_used=0, attempts_last_10_minutes=1
)

### Scenario B: Several suspicious signals

In [ ]:
suspicious_result = check_login(
    login_hour=3, failed_attempts=8, new_device=1,
    distance_from_home_km=500, vpn_used=1, attempts_last_10_minutes=12
)

### Scenario C: Legitimate traveller or ambiguous case

The person is far from their usual location and uses a new device, but has no repeated failures. MFA may be more proportionate than blocking or accusation.

In [ ]:
traveller_result = check_login(
    login_hour=18, failed_attempts=0, new_device=1,
    distance_from_home_km=800, vpn_used=0, attempts_last_10_minutes=1
)

## 12. Your turn

Change the values below. Use only fictional information.

- `login_hour`: 0–23
- `new_device` and `vpn_used`: 0 or 1
- Other values: zero or more

In [ ]:
my_result = check_login(
    login_hour=22, failed_attempts=2, new_device=1,
    distance_from_home_km=120, vpn_used=0, attempts_last_10_minutes=3
)

## 13. Challenge: Can you confuse the model?

Create a scenario that looks suspicious to a human but is classified as normal, or appears normal but is classified as suspicious. Explain the prediction, missing context and a proportionate response.

In [ ]:
challenge_result = check_login(
    login_hour=10, failed_attempts=0, new_device=1,
    distance_from_home_km=700, vpn_used=1, attempts_last_10_minutes=1
)

## 14. Optional AI pair-programming prompts

### Explain code

```text
I am building a suspicious-login detector using Python and a DecisionTreeClassifier.
Explain this cell to a beginner, including the features, label and limitations.
[Paste one cell]
```

### Debug

```text
I am using Google Colab, pandas and scikit-learn.
Error: [paste complete error]
Code: [paste one cell]
Explain the cause, then provide the smallest correction. Do not rewrite the project.
```

### Generate safe test cases

```text
Generate five fictional login scenarios using the six feature names in this notebook.
Include a normal user, repeated-login pattern, traveller, VPN user and ambiguous case.
Use no real personal information.
```

# Final takeaways

1. ML learns patterns from examples.
2. A suspicious prediction is not proof.
3. Accuracy alone can hide important mistakes.
4. False positives and false negatives both matter.
5. Real security requires layered protection.

**ML detection + MFA + rate limiting + alerts + contextual information + human review**